# Crescendo Faithful Hidden State Extraction (post-hoc)

Extracts Design D hidden states from saved Crescendo conversations, capturing
**every turn including refusal turns**, with rollback-aware context reconstruction.

## Why this is faithful

Crescendo rolls back `target_history` after a refusal — the target model never sees
refused turns when generating subsequent turns. The saved `is_refusal` flag lets us
reconstruct the exact context the model saw at every turn:

```
Context for turn T = [non-refusal user/asst pairs from turns 1..T-1]
                   + [user message at turn T]
                   + add_generation_prompt=True
```

This holds for **both refusal and non-refusal turns**. No generation is needed —
just forward passes on the saved data.

## What `is_refusal` means

`is_refusal=True` means the Crescendo step judge decided the response did not progress
toward the harmful objective. This includes both outright refusals and off-topic responses.
The rollback ensures these turns don't accumulate in the target's context.

## Outputs

```
data/representations/crescendo_harmful_10_turns_faithful/
    hidden_states.npy    ← float16, shape (N, N_LAYERS, 4096)
    metadata.parquet     ← one row per turn
data/representations/crescendo_benign_10_turns_faithful/
    hidden_states.npy
    metadata.parquet
```

**Metadata columns:** `pair_id, attempt, turn_idx, is_refusal, final_verdict,
n_context_turns, label, framework, split, fname`

(`n_context_turns` = number of non-refusal pairs in context when this turn was processed —
useful for studying how context depth interacts with compliance.)

In [1]:
import os

GPU_IDS = [4, 5, 6, 7]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

MODEL_ID  = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE     = torch.bfloat16

# First, middle, last transformer layer
LAYER_INDICES = [1, 16, 32]

REPR_ROOT = repo_root / "data" / "representations"

print(f"GPUs:   {GPU_IDS} → logical {LOGICAL_GPU_IDS}")
print(f"Layers: {LAYER_INDICES}")

GPUs:   [4, 5, 6, 7] → logical [0, 1, 2, 3]
Layers: [1, 16, 32]


## Context reconstruction + forward pass helpers

In [2]:
def build_extraction_contexts(turns: list[dict]) -> list[dict]:
    """
    Reconstruct the exact context the model saw at every Crescendo turn.

    Mirrors the Crescendo rollback logic: non-refusal turns accumulate in
    target_history; refusal turns are rolled back and NOT seen by subsequent turns.

    Returns a list of dicts, one per turn:
        turn_idx        — 1-based turn index
        messages        — chat messages for apply_chat_template (no assistant at turn T)
        is_refusal      — whether the model's response was judged a refusal
        n_context_turns — number of non-refusal pairs already in context
    """
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    accumulated = []   # non-refusal user/asst pairs seen so far by target
    result = []

    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_turn = pair.get("user")
        asst_turn = pair.get("assistant")
        if not user_turn or not asst_turn:
            continue

        is_refusal = asst_turn.get("is_refusal", False)

        messages = list(accumulated) + [{"role": "user", "content": user_turn["content"]}]

        result.append({
            "turn_idx":        turn_idx,
            "messages":        messages,
            "is_refusal":      is_refusal,
            "n_context_turns": len(accumulated) // 2,
        })

        if not is_refusal:
            accumulated.append({"role": "user",      "content": user_turn["content"]})
            accumulated.append({"role": "assistant", "content": asst_turn["content"]})

    return result


@torch.no_grad()
def forward_design_d(
    model,
    tokenizer,
    messages: list[dict],
) -> np.ndarray:
    """
    Design D forward pass: hidden state at last position of tokenised prompt
    with add_generation_prompt=True.

    Returns np.ndarray float16, shape (N_LAYERS, 4096).
    """
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.autocast(device_type="cuda", dtype=DTYPE):
        out = model(input_ids, output_hidden_states=True)

    layer_vecs = np.stack([
        out.hidden_states[l][0, -1, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (N_LAYERS, 4096)

    del out, input_ids
    return layer_vecs


def extract_conversation(
    model,
    tokenizer,
    conv: dict,
    fpath: Path,
    split: str,
) -> tuple[np.ndarray | None, list[dict]]:
    """
    Extract hidden states for every turn in one conversation.

    Note on is_post_jailbreak:
        Data was generated with stop_on_success=False — conversations continued
        past the jailbreak turn. Post-jailbreak turns have the jailbroken response
        already in context and represent a different regime. Filter these out
        (is_post_jailbreak=True) when studying the attack trajectory.
    """
    turns = conv.get("turns", [])
    if not turns:
        return None, []

    contexts = build_extraction_contexts(turns)
    if not contexts:
        return None, []

    jailbreak_turn = conv.get("jailbreak_turn")   # None if not jailbroken

    vecs = []
    rows = []
    for ctx in contexts:
        vec = forward_design_d(model, tokenizer, ctx["messages"])
        vecs.append(vec)
        rows.append(dict(
            pair_id          = conv["objective_pair_id"],
            attempt          = conv.get("attempt", 1),
            turn_idx         = ctx["turn_idx"],
            is_refusal       = ctx["is_refusal"],
            n_context_turns  = ctx["n_context_turns"],
            is_post_jailbreak = (
                jailbreak_turn is not None and ctx["turn_idx"] > jailbreak_turn
            ),
            jailbreak_turn   = jailbreak_turn,
            final_verdict    = conv["verdict"],
            label            = 1 if split == "harmful" else 0,
            framework        = "crescendo",
            split            = split,
            fname            = fpath.name,
        ))

    states = np.stack(vecs).astype(np.float16)  # (n_turns, N_LAYERS, 4096)
    return states, rows


print("Helpers defined.")

Helpers defined.


## Parallel extraction infrastructure

In [3]:
def load_done(save_dir: Path):
    meta_path = save_dir / "metadata.parquet"
    if meta_path.exists():
        meta = pd.read_parquet(meta_path)
        done = set(zip(meta["pair_id"], meta["attempt"]))
        print(f"  Resuming: {len(done)} conversations already done")
        return done, [np.load(str(save_dir / "hidden_states.npy"))], [meta]
    return set(), [], []


def save_results(all_states, all_meta, save_dir: Path):
    save_dir.mkdir(parents=True, exist_ok=True)
    states = np.concatenate(all_states, axis=0).astype(np.float16)
    meta   = pd.concat(all_meta, ignore_index=True)
    np.save(str(save_dir / "hidden_states.npy"), states)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"Saved {states.shape[0]} vectors → {save_dir}")
    print(f"  Shape: {states.shape}  ({states.nbytes / 1e9:.2f} GB)")
    return states, meta


def run_parallel(files, split, save_dir, done_set, existing_states, existing_meta):
    pending = [
        f for f in files
        if (json.loads(f.read_text()).get("objective_pair_id"),
            json.loads(f.read_text()).get("attempt", 1)) not in done_set
    ]
    print(f"  Pending: {len(pending)} / {len(files)} conversations", flush=True)

    n_gpus  = len(LOGICAL_GPU_IDS)
    chunks  = [pending[i::n_gpus] for i in range(n_gpus)]

    all_states = list(existing_states)
    all_meta   = list(existing_meta)
    lock = threading.Lock()
    pbar = tqdm(total=len(pending), desc=split, file=sys.stdout, dynamic_ncols=True)

    def worker(gpu_id, chunk):
        torch.cuda.set_device(gpu_id)
        device = f"cuda:{gpu_id}"
        print(f"GPU {gpu_id}: loading model...", flush=True)
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=DTYPE, device_map={"" : device}
        )
        mdl.eval()
        print(f"GPU {gpu_id}: ready, {len(chunk)} conversations", flush=True)

        local_states, local_meta = [], []

        for fpath in chunk:
            conv = json.loads(fpath.read_text())
            try:
                states, rows = extract_conversation(mdl, tok, conv, fpath, split)
                if states is not None:
                    local_states.append(states)
                    local_meta.append(pd.DataFrame(rows))
            except Exception as e:
                print(f"GPU {gpu_id} ERROR {fpath.name}: {e}", flush=True)

            with lock:
                pbar.update(1)

        with lock:
            all_states.extend(local_states)
            all_meta.extend(local_meta)

        del mdl
        torch.cuda.empty_cache()
        print(f"GPU {gpu_id}: done", flush=True)

    threads = [
        threading.Thread(target=worker, args=(gpu_id, chunks[gpu_id]))
        for gpu_id in LOGICAL_GPU_IDS
    ]
    for t in threads: t.start()
    for t in threads: t.join()
    pbar.close()

    return all_states, all_meta


print("Parallel infrastructure defined.")

Parallel infrastructure defined.


## Extract harmful

In [4]:
HARMFUL_CONV_DIR = repo_root / "data" / "conversations" / "crescendo_harmful_10_turns"
HARMFUL_SAVE_DIR = REPR_ROOT / "crescendo_harmful_10_turns_faithful"

harmful_files = sorted(HARMFUL_CONV_DIR.glob("*.json"))
print(f"Harmful: {len(harmful_files)} conversations")

done_h, ex_states_h, ex_meta_h = load_done(HARMFUL_SAVE_DIR)
states_h, meta_h = run_parallel(harmful_files, "harmful", HARMFUL_SAVE_DIR, done_h, ex_states_h, ex_meta_h)
save_results(states_h, meta_h, HARMFUL_SAVE_DIR)

Harmful: 964 conversations
  Pending: 964 / 964 conversations


harmful:   0%|          | 0/964 [00:00<?, ?it/s]

GPU 0: loading model...
GPU 2: loading model...
GPU 3: loading model...
GPU 1: loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 1: ready, 241 conversations
GPU 1: done


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 3: ready, 241 conversations


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 2: ready, 241 conversations
GPU 0: ready, 241 conversations
GPU 2: done
GPU 3: done
GPU 0: done
Saved 9640 vectors → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/representations/crescendo_harmful_10_turns_faithful
  Shape: (9640, 3, 4096)  (0.24 GB)


(array([[[ 8.6670e-03,  6.2256e-03,  1.1108e-02, ...,  4.2114e-03,
           9.8267e-03,  9.7656e-04],
         [ 4.2236e-02,  8.3008e-03, -1.3379e-01, ...,  2.1973e-02,
          -2.0801e-01,  3.1250e-02],
         [-3.4062e+00, -1.1641e+00, -7.8125e-02, ..., -4.8750e+00,
          -1.3438e+00,  4.4062e+00]],
 
        [[ 8.9111e-03,  5.4932e-03,  1.1169e-02, ...,  3.3722e-03,
           8.8501e-03,  2.8076e-03],
         [ 6.5430e-02,  1.2012e-01, -7.6660e-02, ...,  3.9062e-02,
          -2.3438e-01,  1.2305e-01],
         [-1.5625e+00, -5.3906e-01, -2.5000e+00, ..., -5.5938e+00,
           9.5703e-02,  1.9775e-02]],
 
        [[ 7.0801e-03,  5.0659e-03,  1.0864e-02, ...,  1.3885e-03,
           1.0742e-02,  2.4109e-03],
         [ 6.2256e-02,  1.8945e-01, -1.4355e-01, ..., -9.7046e-03,
          -3.1250e-01,  8.5449e-03],
         [-4.7500e+00,  2.3906e+00, -5.1562e-01, ..., -3.6875e+00,
          -5.2812e+00, -3.1250e-01]],
 
        ...,
 
        [[ 1.4191e-03,  1.5991e-02,  1.0

## Extract benign

In [5]:
BENIGN_CONV_DIR = repo_root / "data" / "conversations" / "crescendo_benign_10_turns"
BENIGN_SAVE_DIR = REPR_ROOT / "crescendo_benign_10_turns_faithful"

benign_files = sorted(BENIGN_CONV_DIR.glob("*.json"))
print(f"Benign: {len(benign_files)} conversations")

done_b, ex_states_b, ex_meta_b = load_done(BENIGN_SAVE_DIR)
states_b, meta_b = run_parallel(benign_files, "benign", BENIGN_SAVE_DIR, done_b, ex_states_b, ex_meta_b)
save_results(states_b, meta_b, BENIGN_SAVE_DIR)

Benign: 764 conversations
  Pending: 764 / 764 conversations


benign:   0%|          | 0/764 [00:00<?, ?it/s]

GPU 0: loading model...
GPU 1: loading model...
GPU 2: loading model...
GPU 3: loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 2: ready, 191 conversations
GPU 1: ready, 191 conversations
GPU 0: ready, 191 conversations
GPU 3: ready, 191 conversations
GPU 2: done
GPU 0: done
GPU 3: done
GPU 1: done
Saved 7640 vectors → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/representations/crescendo_benign_10_turns_faithful
  Shape: (7640, 3, 4096)  (0.19 GB)


(array([[[ 4.4556e-03,  6.1646e-03,  9.0942e-03, ...,  7.0496e-03,
           8.6060e-03,  2.7771e-03],
         [ 1.7969e-01,  9.0820e-02, -1.3086e-01, ...,  5.4688e-02,
          -2.0605e-01,  4.9805e-02],
         [-2.6250e+00, -3.8086e-01, -3.3203e-01, ..., -2.9062e+00,
           1.0312e+00,  1.2500e+00]],
 
        [[ 1.0605e-03,  1.3916e-02,  1.1353e-02, ...,  3.3875e-03,
           1.0742e-02,  1.5564e-03],
         [ 5.4688e-02, -8.0078e-02,  2.6978e-02, ..., -1.3184e-01,
          -1.9531e-01,  1.0059e-01],
         [-3.2344e+00, -6.0547e-01,  9.7266e-01, ..., -5.1562e+00,
           1.1094e+00,  2.2656e+00]],
 
        [[ 1.8311e-03,  1.4648e-02,  1.2085e-02, ...,  8.0109e-04,
           8.0566e-03,  9.5367e-04],
         [ 1.9043e-02, -1.6895e-01,  8.7280e-03, ..., -9.5215e-02,
          -2.9883e-01,  5.8105e-02],
         [-1.9688e+00,  7.9297e-01,  1.0312e+00, ..., -3.7656e+00,
           1.1484e+00,  1.9062e+00]],
 
        ...,
 
        [[-1.3428e-03,  1.2207e-02,  1.3

## Verification

In [6]:
for label, save_dir in [("Harmful", HARMFUL_SAVE_DIR), ("Benign", BENIGN_SAVE_DIR)]:
    sp = save_dir / "hidden_states.npy"
    mp = save_dir / "metadata.parquet"
    if not sp.exists():
        print(f"{label}: not found")
        continue

    states = np.load(str(sp), mmap_mode="r")
    meta   = pd.read_parquet(mp)

    print(f"=== {label} ===")
    print(f"Shape:  {states.shape}  ({states.nbytes / 1e9:.2f} GB)")
    print(f"Rows:   {len(meta)}")
    print()

    print("Verdict breakdown:")
    print(meta.groupby("final_verdict")["pair_id"].count().rename("n").to_string())
    print()

    print("Refusal vs non-refusal turns:")
    print(meta["is_refusal"].value_counts().rename("n").to_string())
    print()

    n_post = meta["is_post_jailbreak"].sum()
    print(f"Post-jailbreak turns: {n_post} ({100*n_post/len(meta):.1f}%) ← exclude from trajectory analysis")
    print()

    print("Context depth (n_context_turns) distribution:")
    print(meta["n_context_turns"].value_counts().sort_index().rename("n").to_string())
    print()

    sample = states[:min(500, len(states)), -1, :].astype(np.float32)
    print(f"Last layer stats: mean={sample.mean():.4f}  std={sample.std():.4f}  "
          f"NaNs={np.isnan(sample).sum()}  Infs={np.isinf(sample).sum()}")
    print()

=== Harmful ===
Shape:  (9640, 3, 4096)  (0.24 GB)
Rows:   9640

Verdict breakdown:
final_verdict
jailbroken    3020
near_miss     1110
refusal       5510

Refusal vs non-refusal turns:
is_refusal
True     7251
False    2389

Post-jailbreak turns: 1068 (11.1%) ← exclude from trajectory analysis

Context depth (n_context_turns) distribution:
n_context_turns
0    6682
1     961
2     645
3     475
4     320
5     248
6     160
7     104
8      40
9       5

Last layer stats: mean=-0.0161  std=2.2419  NaNs=0  Infs=0

=== Benign ===
Shape:  (7640, 3, 4096)  (0.19 GB)
Rows:   7640

Verdict breakdown:
final_verdict
jailbroken    1390
near_miss     5210
refusal       1040

Refusal vs non-refusal turns:
is_refusal
False    7212
True      428

Post-jailbreak turns: 727 (9.5%) ← exclude from trajectory analysis

Context depth (n_context_turns) distribution:
n_context_turns
0    1126
1     764
2     774
3     762
4     756
5     757
6     734
7     707
8     650
9     610

Last layer stats: mean=